# Soft Attention (软注意力) 实现

软注意力是最基础的注意力机制形式，对所有位置计算权重，整个过程可微分。

## 1. 导入必要的库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 2. 实现Softmax函数

In [ ]:
def softmax(x):
    """Softmax函数实现"""
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

## 3. 实现软注意力类

In [ ]:
class SoftAttention:
    """软注意力机制实现类"""
    
    def __init__(self, hidden_dim):
        self.hidden_dim = hidden_dim
        # 初始化权重矩阵
        self.W_a = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.W_c = np.random.randn(hidden_dim, 1) * 0.01
        self.b_a = np.zeros((hidden_dim,))
    
    def forward(self, query, keys, values):
        """前向传播"""
        seq_len = keys.shape[0]
        
        # 计算注意力得分
        scores = []
        for i in range(seq_len):
            combined = query + keys[i]
            hidden = np.tanh(np.dot(combined, self.W_a) + self.b_a)
            score = np.dot(hidden, self.W_c.flatten())
            scores.append(score)
        
        scores = np.array(scores)
        
        # Softmax归一化
        attention_weights = softmax(scores)
        
        # 加权求和
        context = np.zeros(self.hidden_dim)
        for i in range(seq_len):
            context += attention_weights[i] * values[i]
        
        return context, attention_weights

## 4. 简化版软注意力实现

In [ ]:
def soft_attention_simple(query, keys, values):
    """简化版软注意力实现"""
    # 计算注意力得分（点积）
    scores = np.dot(keys, query)
    
    # Softmax归一化
    weights = softmax(scores)
    
    # 加权求和
    output = np.dot(weights, values)
    
    return output, weights

## 5. 测试软注意力机制

In [ ]:
# 参数设置
hidden_dim = 64
seq_len = 10

# 创建软注意力层
attention = SoftAttention(hidden_dim)

# 生成示例数据
query = np.random.randn(hidden_dim)
keys = np.random.randn(seq_len, hidden_dim)
values = np.random.randn(seq_len, hidden_dim)

# 前向传播
context, weights = attention.forward(query, keys, values)

print(f"查询向量形状: {query.shape}")
print(f"键向量形状: {keys.shape}")
print(f"值向量形状: {values.shape}")
print(f"\n上下文向量形状: {context.shape}")
print(f"注意力权重形状: {weights.shape}")
print(f"\n注意力权重: {weights}")
print(f"注意力权重总和: {np.sum(weights):.6f}")

## 6. 可视化注意力权重

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(range(seq_len), weights)
plt.xlabel('位置索引')
plt.ylabel('注意力权重')
plt.title('软注意力权重分布')
plt.grid(True, alpha=0.3)
plt.show()

## 7. 测试简化版软注意力

In [ ]:
output, simple_weights = soft_attention_simple(query, keys, values)

print(f"输出向量形状: {output.shape}")
print(f"注意力权重: {simple_weights}")
print(f"注意力权重总和: {np.sum(simple_weights):.6f}")

# 可视化
plt.figure(figsize=(10, 4))
plt.bar(range(seq_len), simple_weights)
plt.xlabel('位置索引')
plt.ylabel('注意力权重')
plt.title('简化版软注意力权重分布')
plt.grid(True, alpha=0.3)
plt.show()